# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking
because the preoblem requires us to make a priority list of which pages need to be reviewed first, therefore they need to be ranked from top to bottom from highest to lowest priority.This helps us save the human content reviewer's time and get the most important work done first. 

In [10]:
import pandas as pd 
df = pd.read_csv("d:/git/Assignment-1/data/raw/content_refresh_anonymized.csv")
print("rows:", df.shape[0])
print('unique content items:', df['content_id'].nunique())
print('unique clients:', df['client_id'].nunique())
print('trend counts:', df["trend_direction"].value_counts())

df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "sessions_90d",
        "content_age_days",
        "trend_direction",
        "trend_pct",

    ]
].head(10)


rows: 30000
unique content items: 30000
unique clients: 32
trend counts: trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,3803,17,187,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down,-60.9
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down,-38.9
6,content_9a34b442b552,client_8722616204,20,1,90,down,-92.3
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down,-58.8
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down,-29.2


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: I will creat a priority list and scores of pages to be refreshed in order to improve their performance.I am using the proxy label based on current trend direction for this as the true future outcome is not available. This makes it a reasonable starting point but not the final ideal target.

In [11]:
preview_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "trend_direction",
    "trend_pct",
]

df[preview_cols].head(10)

,content_id,client_id,impressions_90d,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,3803,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,15320,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,12581,down,-60.9
3,content_331d6c4de07b,client_19581e27de,11751,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,19140,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,3970,down,-38.9
6,content_9a34b442b552,client_8722616204,20,down,-92.3
7,content_a63219c6e95a,client_19581e27de,1724,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,32574,down,-58.8
9,content_c27558df2b0c,client_19581e27de,1240,down,-29.2


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use precision@K, or to be specific, precision@50 as the success metric to get a score for how well my work actually lists the true positive pages at higher priority scores. 

In [12]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

df["score"] =(
    df["impressions_90d"] * 0.1
    + df["sessions_90d"] * 0.3
    - df["avg_position"] * 0.1
)
top50 = df.sort_values("score", ascending=False).head(50)
precision_at_50 = top50["is_declining"].mean()
print(f"Precision@50 = {precision_at_50:.3f}")

Precision@50 = 0.420


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

one row= one content item

In [13]:
print("One row here is one content item.")
print("Top columns for the unit of analysis:")
df[preview_cols].head(10)

One row here is one content item.
Top columns for the unit of analysis:


,content_id,client_id,impressions_90d,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,3803,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,15320,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,12581,down,-60.9
3,content_331d6c4de07b,client_19581e27de,11751,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,19140,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,3970,down,-38.9
6,content_9a34b442b552,client_8722616204,20,down,-92.3
7,content_a63219c6e95a,client_19581e27de,1724,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,32574,down,-58.8
9,content_c27558df2b0c,client_19581e27de,1240,down,-29.2


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The pattern is too messy because it depends on multiple signle simultaneously making a simple if else strategy too simple to address non-linear and context dependent cases.

In [14]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
#a simple threshold rule using if else strategy is too weak 
rule = df["impressions_90d"] < 1000
print("Simple rule: impressions_90d < 1000")
print("Precision:", df.loc[rule, "is_declining"].mean())
print("Coverage:", rule.mean())
print()
#in reality it is better to use a machine learning model to learn the best combination of features and thresholds to predict decline
for col in ["impressions_90d", "sessions_90d", "avg_position", "ctr"]:
    print("===", col, "===")
    print(
        df.groupby(pd.cut(df[col], 5))["is_declining"]
        .mean()
        .rename("decline_rate")
    )
    print()

Simple rule: impressions_90d < 1000
Precision: 0.49921154779233384
Coverage: 0.5496

=== impressions_90d ===
impressions_90d
(-516.714, 103543.8]    0.542901
(103543.8, 207086.6]    0.370370
(207086.6, 310629.4]    0.473684
(310629.4, 414172.2]    0.333333
(414172.2, 517715.0]    0.571429
Name: decline_rate, dtype: float64

=== sessions_90d ===
sessions_90d
(-3.344, 869.8]     0.542383
(869.8, 1738.6]     0.422535
(1738.6, 2607.4]    0.500000
(3476.2, 4345.0]    0.333333
Name: decline_rate, dtype: float64

=== avg_position ===
avg_position
(-0.245, 49.0]    0.551686
(49.0, 98.0]      0.349928
(98.0, 147.0]     0.071429
(147.0, 196.0]    0.000000
(196.0, 245.0]    0.000000
Name: decline_rate, dtype: float64

=== ctr ===
ctr
(-0.1, 20.0]     0.543920
(20.0, 40.0]     0.180851
(40.0, 60.0]     0.031250
(60.0, 80.0]     0.500000
(80.0, 100.0]    0.090909
Name: decline_rate, dtype: float64



This output shows decline is not isolated to one simple threshold, pages with low impressions are not the only ones that decline and the decline rate changes across position, sessions, and CTR. That is why a model that combines multiple signals is more appropriate than one fixed if else rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.